In [11]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

In [ ]:
load_dotenv()  # Load environment variables from .env file

In [ ]:
model = ChatOpenAI(model_name="gpt-4", temperature=0.7)

In [ ]:
class BlogState(TypedDict):
    topic: str
    outline: str
    content: str
    evaluation: str

In [ ]:
def create_outline(state: BlogState) -> BlogState:
    title = state['topic']
    prompt = f"Create a detailed outline for a blog post about '{title}'."
    outline = model.invoke(prompt)
    state['outline'] = outline
    return state

In [ ]:
def create_blog(state: BlogState) -> BlogState:
    title = state['topic']
    outline = state['outline']

    prompt = f"write a detailed blog on the title {title} with the following outline:\n{outline}"
    content = model.invoke(prompt)
    state['content'] = content
    return state

In [ ]:
def evaluate_blog(state:BlogState)->BlogState:
    outline = state['outline']
    score = model.invoke(f"Evaluate the quality of the following blog outline:\n{outline}\nProvide a score from 1 to 10 and a brief explanation.")
    state['evaluation'] = score
    return state

In [ ]:
graph = StateGraph(BlogState)

# nodes
graph.add_node("create_outline", create_outline)
graph.add_node("create_blog",create_blog)
graph.add_node("evaluate_blog",evaluate_blog)
# edges
graph.add_edge(START, "create_outline")
graph.add_edge("create_outline", "create_blog")
graph.add_edge("create_blog", "evaluate_blog")
graph.add_edge("evaluate_blog", END)

workflow = graph.compile()

In [ ]:
initial_state = {"topic": "The Future of AI"}
final_state = workflow.invoke(initial_state)
print(final_state)